# Setup

In [ ]:
import os
import json
from dotenv import load_dotenv
from upstash_vector import Index
from google import genai as google_genai
import time
from datetime import datetime

data_atual = datetime.now().strftime("%d/%m/%Y")

load_dotenv()

UPSTASH_ENDPOINT = os.getenv("UPSTASH_ENDPOINT")
UPSTASH_API_KEY = os.getenv("UPSTASH_API_KEY")
GEMINI_API_KEY_T1 = os.getenv("GEMINI_API_KEY_T1")
FALLBACK_URL = "https://ifrs.edu.br/canoas/"

index = Index(url=UPSTASH_ENDPOINT, token=UPSTASH_API_KEY)
google_client = google_genai.Client(api_key=GEMINI_API_KEY_T1)

# carrega agentes
AGENT_NAME = "agente-ifrs"

with open(f"../data/info/{AGENT_NAME}.txt", "r", encoding="utf-8") as f:
    agent_prompt = f.read()

print("Conexões configuradas.")

# RAG

In [ ]:
ALPHA = 0.7
MIN_YEAR = 2020

def rerank_by_date(hits):
    max_year = datetime.now().year

    def date_score(hit):
        raw = hit.metadata.get("published_at")
        if not raw:
            return 0.5
        try:
            year = int(raw)
            return (year - MIN_YEAR) / (max_year - MIN_YEAR)
        except (ValueError, TypeError):
            return 0.5

    return sorted(
        hits,
        key=lambda h: ALPHA * h.score + (1 - ALPHA) * date_score(h),
        reverse=True
    )


def search(query, top_k=10):
    for attempt in range(3):
        try:
            result = google_client.models.embed_content(
                model="gemini-embedding-001",
                contents=query
            )
            vector = result.embeddings[0].values
            hits = index.query(
                vector=vector,
                top_k=top_k,
                include_metadata=True
            )
            return hits
        except Exception as e:
            if "429" in str(e):
                wait = 30 * (attempt + 1)
                print(f"  Rate limit embedding, aguardando {wait}s...")
                time.sleep(wait)
            else:
                raise e
    return []


def build_context(hits, min_score=0.60):
    filtered = [h for h in hits if h.score >= min_score]
    
    context = ""
    sources = {}
    seen_urls = {}
    counter = 1
    
    for h in filtered:
        url = h.metadata['source_url']
        
        # deduplica URLs no índice de fontes
        if url not in seen_urls:
            seen_urls[url] = counter
            sources[counter] = url
            counter += 1
        
        source_num = seen_urls[url]
        published_at = h.metadata.get('published_at') or 'data desconhecida'
        context += f"[{source_num}] Fonte: {url} | Data: {published_at}\n"
        context += h.metadata['text'] + "\n\n"
    
    return context, filtered, sources


def ask(query, history=None, top_k=15):
    if history is None:
        history = []

    # monta histórico antes de qualquer caminho
    history_text = ""
    for msg in history:
        role = "Estudante" if msg["role"] == "user" else "Assistente"
        history_text += f"{role}: {msg['content']}\n"

    # search e construção de contexto, usa apenas as últimas 2 perguntas para o search
    recent_questions = [msg for msg in history if msg["role"] == "user"][-2:]
    recent_history_text = ""
    for msg in recent_questions:
        recent_history_text += f"Estudante: {msg['content']}\n"

    if recent_history_text:
        search_query = f"Data atual: {data_atual}\n{recent_history_text}\nPergunta atual: {query}" if recent_history_text else f"Data atual: {data_atual}\n{query}"
    else:
        search_query = query

    # busca vetorial
    hits = search(search_query, top_k=top_k)
    hits = rerank_by_date(hits)
    context, filtered, sources = build_context(hits)

    print(f"Chunks encontrados: {len(filtered)}")

    # se nao achou chunks, fallback para pesquisa na internet
    if not filtered:
        response = google_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=f"Você é um assistente do IFRS Campus Canoas.\n\nHistórico:\n{history_text}\n\nResponda a seguinte pergunta buscando na internet, priorizando fontes do IFRS: {query}",
            config={"tools": [{"google_search": {}}], "temperature": 0.7},
        )

        result = response.text

        try:
            chunks = response.candidates[0].grounding_metadata.grounding_chunks
            for c in chunks:
                if c.web:
                    result += "\n\n*Esta resposta foi obtida por busca na internet, pois não encontrei o conteúdo na base de documentos do campus.*"
                    break
        except Exception:
            pass

        return result

    # se achou, gera a query
    sources_text = "\n".join([f"[{i}] {url}" for i, url in sources.items()])
    prompt = agent_prompt.format(
        context=context,
        query=query,
        sources=sources_text,
        history=history_text,
        data_atual=data_atual
    )

    response = google_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={"temperature": 0.7}
    )
    return response.text

# Teste individual

In [ ]:
# teste
print(ask("Quais disciplinas o Rafael Coimbra Pinto leciona?"))

Chunks encontrados: 15
Rafael Pinto leciona Est. Dados no LAB E10 (INF) no TADS 3º semestre [1].
Rafael Pinto leciona Desenv. Web I no LAB E09 (INF) no TADS 4º semestre [1].
Rafael Pinto leciona IA no LAB D06 (INF) no TADS 6º semestre [1].
Rafael Pinto leciona Tops. Esp. Comp. no LAB D06 (INF) no TADS 6º semestre [1].

Fontes:
[1] https://drive.google.com/file/d/1pQIT_vhyxsq49OJ8N71owZUBTj21Hxzo/view


In [3]:
# teste
print(ask("Quem da aula de tópicos especiais em computação?"))

Chunks encontrados: 15
Rafael Pinto leciona Tópicos Especiais em Computação no LAB D06 (INF) para o TADS 6º semestre [1].

Fontes:
[1] https://drive.google.com/file/d/1pQIT_vhyxsq49OJ8N71owZUBTj21Hxzo/view


In [6]:
# teste
print(ask("Quem é o professor de Desenvolvimento Web 2?"))

Chunks encontrados: 15
O professor de Desenvolvimento Web 2 é Márcio Bigolin [5].

Fontes:
[5] https://drive.google.com/file/d/1u16bttceP7Hf1ampA-YeO7p3KF5hYa2h/view


In [ ]:
hits = search("Quem da aula de tópicos especiais em computação?", top_k=15)
for h in hits[:5]:
    print(f"{h.score:.4f} - {h.metadata['source_url']}")

In [7]:
# teste
print(ask("Quem são os coordenadores dos cursos?"))

Chunks encontrados: 15
O IFRS Campus Canoas possui os seguintes coordenadores de curso:

Técnico em Administração Integrado ao Ensino Médio: Leila de Almeida Castillo [1, 5, 7]
Técnico em Desenvolvimento de Sistemas Integrado ao Ensino Médio: Aline Noimann [1, 5, 7]
Técnico em Eletrônica Integrado ao Ensino Médio: Marianna da Silva Rogério Mussatto [1, 5, 7]
Técnico em Comércio Integrado ao Ensino Médio – modalidade Educação de Jovens e Adultos: Carlos Alencar Souza Alves Junior [1, 5, 7]
Superior de Matemática – Licenciatura: Claudiomir Feustler Rodrigues de Siqueira [1, 5, 7]
Superior de Tecnologia em Análise e Desenvolvimento de Sistemas: Ígor Lorenzato Almeida [1, 5, 7]
Superior de Tecnologia em Automação Industrial: Emílio Rodolfo Arend [1, 5, 7]
Superior de Tecnologia em Logística: Evandro Nascimento [1, 5, 7]
Superior de Engenharia Eletrônica: Joel Augusto Luft [1, 5, 7]
Pós-Graduação em Linguagens Contemporâneas e Ensino: Sheila Katinae Staudt [1]
Pós-Graduação em Gestão de Pro

# Teste com historico

In [11]:
history = []

while True:
    query = input("Você: ")
    if query.lower() in ["sair", "exit", "quit"]:
        break
    
    print(f"\nVocê: {query}")
    
    resposta = ask(query, history=history)
    history.append({"role": "user", "content": query})
    history.append({"role": "assistant", "content": resposta})
    
    print(f"\nAssistente: {resposta}")
    print("-" * 50)


Você: quem é voce?
Chunks encontrados: 15

Assistente: Eu sou o assistente virtual do IFRS Campus Canoas, criado para ajudar estudantes.
--------------------------------------------------

Você: quem da aula de lpoo1?
Chunks encontrados: 15

Assistente: A disciplina "Linguagem de Programação Orientada a Objetos I" (LPOO I) é lecionada por Jair Azevedo no TADS (Técnico em Automação Industrial) no 3º semestre [1].

Fontes:
[1] https://drive.google.com/file/d/1pQIT_vhyxsq49OJ8N71owZUBTj21Hxzo/view
--------------------------------------------------

Você: quem é rafael pinto?
Chunks encontrados: 15

Assistente: Rafael Pinto leciona as seguintes disciplinas:

Estudos dos Dados no TADS (Técnico em Automação Industrial) no 3º semestre [1].
Desenvolvimento Web I no TADS no 4º semestre [1].
Inteligência Artificial (IA) no TADS no 6º semestre [1].
Tópicos Especiais em Computação (Tops. Esp. Comp.) no TADS no 6º semestre [1].

Fontes:
[1] https://drive.google.com/file/d/1pQIT_vhyxsq49OJ8N71owZUB

In [ ]:
from datetime import datetime

data_atual = datetime.now().strftime("%d/%m/%Y")

search_query = f"Data atual: {data_atual}\n{recent_history_text}\nPergunta atual: {query}" if recent_history_text else f"Data atual: {data_atual}\n{query}"

[{'role': 'user', 'content': 'quem é voce?'},
 {'role': 'assistant',
  'content': 'Eu sou o assistente virtual do IFRS Campus Canoas, criado para ajudar estudantes.'},
 {'role': 'user',
  'content': 'quem da aula de topicos especiais em computação?'},
 {'role': 'assistant',
  'content': 'Rafael Pinto leciona Tópicos Especiais em Computação no TADS no 6º semestre [1].\n\nFontes:\n[1] https://drive.google.com/file/d/1pQIT_vhyxsq49OJ8N71owZUBTj21Hxzo/view'},
 {'role': 'user', 'content': 'quem da aula de LPOO1?'},
 {'role': 'assistant',
  'content': 'Jair Azevedo leciona LPOO1 no TADS no 3º semestre [1].\n\nFontes:\n[1] https://drive.google.com/file/d/1pQIT_vhyxsq49OJ8N71owZUBTj21Hxzo/view'},
 {'role': 'user', 'content': 'que dia termina esse semestre?'},
 {'role': 'assistant',
  'content': 'Não encontrei informações sobre a data de término do semestre nos documentos disponíveis.\n\nFontes:\n[]'}]

In [ ]:
hits = search("Tópicos Especiais em Computação professor TADS")
for h in hits:
    print(f"{h.score:.4f} - {h.metadata['source_url']}")

In [8]:
print(ask("e de tópicos especiais em computação?", history=[
    {"role": "user", "content": "quem da aula de LPOO 1?"},
    {"role": "assistant", "content": "A disciplina de LPOO 1 não foi encontrada nos documentos disponíveis."}
]))

Chunks encontrados: 15
A disciplina de Tópicos Especiais em Computação foi deferida. [2]

Fontes:
[2] https://ifrs.edu.br/canoas/wp-content/uploads/sites/6/2025/01/COMPLEMENTO-AO-EDITAL-01_2025-CERTIFICACAO-DE-CONHECIMENTOS-HOMOLOGACAO-DAS-SOLICITACOES-E-CRONOGRAMA-DE-APLICACAO-DAS-PROVAS.pdf


In [ ]:
hits = search("quem da aula de LPOO 1\nPergunta atual: e de tópicos especiais em computação?", top_k=15)
for h in hits:
    print(f"{h.score:.4f} - {h.metadata['source_url']}")

In [ ]:
hits = search("quem da aula de LPOO 1\nPergunta atual: e de tópicos especiais em computação?", top_k=15)
for h in hits:
    if "1pQIT" in h.metadata["source_url"]:
        print(h.metadata["text"][:500])

In [ ]:
for h in hits:
    if "1pQIT" in h.metadata["source_url"]:
        print(h.metadata["text"])
        print("---")